# Analyze surveychat transcripts with Python

This short notebook reads a Qualtrics CSV file and turns the surveychat JSON transcript column into a table with one row per chat message.

Only edit the two variables in the first code cell: the CSV file name and the column that contains the JSON transcript.

In [ ]:
import json

import pandas as pd

# Edit these two lines.
csv_file = "qualtrics_export.csv"
json_column = "chat_transcript"

responses = pd.read_csv(csv_file)

if json_column not in responses.columns:
    available = ", ".join(responses.columns)
    raise ValueError(f"Column not found. Check json_column. Available columns: {available}")

# Keep only rows where the transcript column is not missing or blank.
transcripts = responses[json_column].dropna().astype(str).str.strip()
transcripts = transcripts[transcripts.ne("")]

print(f"Found {len(transcripts)} non-empty transcript row(s).")

## Parse the JSON transcripts

The result is `messages`: one row per message, with the response number, turn number, role, content, timestamp, and word count.

In [ ]:
all_messages = []
parse_errors = []

for row_index, transcript_json in transcripts.items():
    response_number = row_index + 1

    try:
        transcript = json.loads(transcript_json)
    except json.JSONDecodeError:
        parse_errors.append(response_number)
        continue

    for turn_number, message in enumerate(transcript.get("messages", []), start=1):
        all_messages.append(
            {
                "response_number": response_number,
                "turn_number": turn_number,
                "role": message.get("role"),
                "content": message.get("content", ""),
                "timestamp": message.get("timestamp"),
            }
        )

messages = pd.DataFrame(all_messages)

if messages.empty:
    raise ValueError("No messages were parsed. Check csv_file and json_column.")

messages["content"] = messages["content"].fillna("").astype(str)
messages["timestamp"] = pd.to_datetime(messages["timestamp"], utc=True, errors="coerce")
messages["word_count"] = messages["content"].str.split().str.len()
messages["participant_word_count"] = messages["word_count"].where(
    messages["role"].eq("participant"),
    0,
)
messages["assistant_word_count"] = messages["word_count"].where(
    messages["role"].eq("assistant"),
    0,
)

if parse_errors:
    print(f"{len(parse_errors)} row(s) could not be parsed. First few: {parse_errors[:10]}")
else:
    print("All non-empty transcript rows parsed successfully.")

messages.head()

## Simple conversation metrics

`conversation_summary` gives one row per respondent. These are simple starting points, not required analyses.

In [ ]:
conversation_summary = (
    messages.groupby("response_number")
    .agg(
        n_turns=("turn_number", "count"),
        participant_turns=("role", lambda x: x.eq("participant").sum()),
        assistant_turns=("role", lambda x: x.eq("assistant").sum()),
        participant_words=("participant_word_count", "sum"),
        assistant_words=("assistant_word_count", "sum"),
        duration_minutes=(
            "timestamp",
            lambda x: (x.max() - x.min()).total_seconds() / 60 if x.notna().sum() >= 2 else pd.NA,
        ),
    )
    .reset_index()
)

conversation_summary.head()

In [ ]:
role_summary = (
    messages.groupby("role")
    .agg(
        n_messages=("content", "count"),
        mean_words=("word_count", "mean"),
    )
    .reset_index()
)

role_summary

## Simple text example

This combines all participant messages from each response. You can use this table for hand coding, keyword checks, or later text analysis.

In [ ]:
participant_text = (
    messages[messages["role"].eq("participant")]
    .groupby("response_number")["content"]
    .apply(lambda parts: " ".join(parts.dropna()))
    .reset_index(name="participant_text")
)

participant_text.head()

In [ ]:
# Change this word to match your research question.
keyword = "trust"

keyword_hits = participant_text["participant_text"].str.lower().str.contains(
    keyword.lower(),
    na=False,
)

keyword_summary = pd.DataFrame(
    {
        "n_responses": [len(participant_text)],
        "responses_mentioning_keyword": [keyword_hits.sum()],
        "percent_mentioning_keyword": [keyword_hits.mean() * 100],
    }
)

keyword_summary